## Multi-Model Translation Comparison (Bengali $\to$ German)
This notebook executes a zero-shot translation benchmark to evaluate how well the **Odia-fine-tuned** models generalize to a related high-resource Indic language (**Bengali**). It serves as a crucial "cross-lingual transfer" test.

### Key Achievements:
* **Zero-Shot Evaluation:** Tested whether the NLLB models, which were fine-tuned only on Odia-German data, retained their ability to translate **Bengali**, a sister language to Odia.
* **Model Lineup:**
  1. **Baseline NLLB:** Pre-trained multilingual NLLB model
  2. **Full Fine-Tuned (FFT) NLLB:** Specialized full fine-tuned NLLB model
  3. **LoRA Fine-Tuned NLLB:** Parameter-efficient NLLB model
  4. **Google Translate**

### Workflow Context:
* **Input:** `bengali_news_sentences_inference.jsonl` (Subset N=50)
* **Process:** Zero-Shot Inference (Target: `deu_Latn`)
* **Output:** `bengali_to_german_model_translation_comparison_results.csv`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.7 MB/s eta 0:00:00


In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from peft import PeftModel
from deep_translator import GoogleTranslator
from tqdm import tqdm
import time

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
# Input Data
INPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/bengali_sentences_inference_50.jsonl"
OUTPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/eval/bengali_to_german_model_translation_comparison_results.csv"

# Model Paths
FULL_FT_MODEL_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/nllb-odia-german-translator_model_final_v2"
LORA_ADAPTER_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/lora-odia-german-translator_v2"
BASE_MODEL_NAME = "facebook/nllb-200-distilled-600M"

# --- Language & Prefix ---
SOURCE_LANG_CODE = "bn"          # Source language code for Google Translate/Internal use
TARGET_LANG_CODE = "de"          # Target language code for Google Translate/Internal use
NLLB_SRC_LANG = "ben_Beng"       # NLLB's source language tag for Bengali
NLLB_TGT_LANG = "deu_Latn"       # NLLB's target language tag for German
PREFIX_TO_STRIP = "translate Bengali to German: "

In [ ]:
# --- Device Setup ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# Load the data
print("Loading dataset...")
records = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

records = records[:50]
print(f"Data slice complete. Processing {len(records)} sentences.")
print(f"First sentence:\n{records[0]}")

Loading dataset...
Data slice complete. Processing 50 sentences.
First sentence:
{'id': 1, 'input_text': 'translate Bengali to German: গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ বৃহস্পতিবার রাতের টিফিন খেয়ে একটি পোশাক কারখানার ৫০০ শ্রমিক অসুস্থ হয়ে পড়েছেন।', 'raw_bengali': 'গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ বৃহস্পতিবার রাতের টিফিন খেয়ে একটি পোশাক কারখানার ৫০০ শ্রমিক অসুস্থ হয়ে পড়েছেন।', 'source': 'unknown', 'original_paragraph_snippet': 'গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ ...'}


In [ ]:
# ==============================================================================
# 2. MODEL AND DATA LOADING
# ==============================================================================

# --- Load Base Model, Tokenizer, and Fine-Tuned Weights ---
print("\n--- 1/4 Loading Base Model and Tokenizer ---")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    source_lang=NLLB_SRC_LANG,
    target_lang=NLLB_TGT_LANG
)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME).to(DEVICE)


--- 1/4 Loading Base Model and Tokenizer ---


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
# --- LoRA Model Loading ---
print("\n--- 2/4 Loading LoRA Model ---")
# Instantiate LoRA model by attaching adapter to the base model
lora_model = PeftModel.from_pretrained(
    base_model,
    LORA_ADAPTER_PATH,
    adapter_name="lora_adapter"
).to(DEVICE)
lora_model.eval()


--- 2/4 Loading LoRA Model ---


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): M2M100ForConditionalGeneration(
      (model): M2M100Model(
        (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
        (encoder): M2M100Encoder(
          (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
          (embed_positions): M2M100SinusoidalPositionalEmbedding()
          (layers): ModuleList(
            (0-11): 12 x M2M100EncoderLayer(
              (self_attn): M2M100Attention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (lora_adapter): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (lora_adapter): Linear(in_features=1024, out_features=16, bias=False)
                  )
           

In [ ]:
# --- Full Fine-Tuned Model Loading ---
print("\n--- 3/4 Loading Full Fine-Tuned Model ---")
full_ft_model = AutoModelForSeq2SeqLM.from_pretrained(FULL_FT_MODEL_PATH).to(DEVICE)
full_ft_model.eval()


--- 3/4 Loading Full Fine-Tuned Model ---


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [ ]:
# ==============================================================================
# 3. TRANSLATION FUNCTIONS
# ==============================================================================

def generate_nllb_translation(model, source_text_list, tokenizer, device):
    """
    Generates translations using a pre-loaded NLLB model with beam search.

    This function processes a list of source sentences by:
    1. Stripping the training-specific prefix (defined globally as `PREFIX_TO_STRIP`).
    2. Tokenizing and moving inputs to the specified PyTorch device.
    3. Generating translations using Beam Search (k=5) to ensure high-quality output.
    4. Enforcing the target language via `forced_bos_token_id`.

    Args:
        model (torch.nn.Module): The loaded NLLB model (e.g., AutoModelForSeq2SeqLM).
        source_text_list (List[str]): A list of source sentences to translate.
        tokenizer (PreTrainedTokenizer): The tokenizer associated with the model.
        device (torch.device): The device (CPU or CUDA) where the model is loaded.

    Global Dependencies:
        - `PREFIX_TO_STRIP` (str): Prefix to remove from input text (e.g., "translate X to Y:").
        - `NLLB_TGT_LANG` (str): The specific target language code (e.g., "deu_Latn").

    Returns:
        List[str]: A list of translated sentences corresponding to the input list.
    """
    translated_texts = []

    # 1. Strip the prefix added in the data preparation step
    stripped_texts = [text.replace(PREFIX_TO_STRIP, "") for text in source_text_list]

    for text in tqdm(stripped_texts, desc="NLLB Batch"):
        # Set max_length to a reasonable value for sentence translation
        # Use beam search for higher quality
        input_ids = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **input_ids,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(NLLB_TGT_LANG),
                max_length=150,
                num_beams=5,
                early_stopping=True
            )

        translated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        translated_texts.append(translated_text)

    return translated_texts

def translate_google_deep_translator(source_text_list, src_lang, tgt_lang):
    """
    Performs batch translation using the Google Translate API via `deep_translator`.

    This wrapper handles batching to respect API character limits and includes rate
    limiting to prevent IP bans. It also performs specific text cleaning (removing
    a hardcoded Bengali-German prefix).

    Args:
        source_text_list (List[str]): The list of sentences to translate.
        src_lang (str): The source language code (e.g., 'bn', 'auto').
        tgt_lang (str): The target language code (e.g., 'de').

    Returns:
        List[str]: A list of translated strings. If a batch fails, the corresponding
        entries are filled with the string "ERROR" to maintain index alignment.
    """
    # Initialize the translator
    # Note: deep_translator uses 'auto' or standard codes like 'en', 'de', 'bn'
    translator = GoogleTranslator(source=src_lang, target=tgt_lang)

    translations = []

    # 1. Strip the prefix (if your texts still have them)
    PREFIX_TO_STRIP = "translate Bengali to German: "
    stripped_texts = [text.replace(PREFIX_TO_STRIP, "") for text in source_text_list]

    # 2. Batch Processing
    # deep_translator is robust, but for 50+ items, it's safer to do small batches
    # to avoid hitting Google's character limit per request.
    batch_size = 10

    print(f"Translating {len(stripped_texts)} sentences using deep_translator...")

    for i in range(0, len(stripped_texts), batch_size):
        batch = stripped_texts[i : i + batch_size]
        try:
            # deep_translator accepts a list of strings directly!
            batch_translations = translator.translate_batch(batch)
            translations.extend(batch_translations)

            # Gentle sleep to be polite to the API
            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error in batch {i}: {e}")
            # Fallback: add empty strings or error markers so lists stay aligned
            translations.extend(["ERROR"] * len(batch))

    return translations

In [ ]:
# ==============================================================================
# 4. EXECUTE TRANSLATION
# ==============================================================================
input_texts = [r['input_text'] for r in records]

# --- 1. Baseline NLLB ---
print("\n--- 1/4 Generating Baseline NLLB Translations ---")
base_translations = generate_nllb_translation(base_model, input_texts, tokenizer, DEVICE)
for i, t in enumerate(base_translations):
    records[i]['nllb_base_translation'] = t

# --- 2. Full Fine-Tuned NLLB ---
print("\n--- 2/4 Generating Full Fine-Tuned NLLB Translations ---")
ft_translations = generate_nllb_translation(full_ft_model, input_texts, tokenizer, DEVICE)
for i, t in enumerate(ft_translations):
    records[i]['nllb_full_ft_translation'] = t

# --- 3. LoRA Fine-Tuned NLLB ---
print("\n--- 3/4 Generating LoRA NLLB Translations ---")
lora_translations = generate_nllb_translation(lora_model, input_texts, tokenizer, DEVICE)
for i, t in enumerate(lora_translations):
    records[i]['nllb_lora_translation'] = t

# --- 4. Google Translate ---
print("\n--- 4/4 Generating Google Translations (deep_translator) ---")
google_translations = translate_google_deep_translator(input_texts, SOURCE_LANG_CODE, TARGET_LANG_CODE)
for i, t in enumerate(google_translations):
    records[i]['google_translation'] = t


--- 1/4 Generating Baseline NLLB Translations ---


NLLB Batch: 100%|██████████| 50/50 [00:31<00:00,  1.57it/s]



--- 2/4 Generating Full Fine-Tuned NLLB Translations ---


NLLB Batch: 100%|██████████| 50/50 [00:22<00:00,  2.23it/s]



--- 3/4 Generating LoRA NLLB Translations ---


NLLB Batch: 100%|██████████| 50/50 [00:30<00:00,  1.63it/s]



--- 4/4 Generating Google Translations (deep_translator) ---
Translating 50 sentences using deep_translator...


In [ ]:
# ==============================================================================
# 5. SAVE RESULTS
# ==============================================================================

print(f"\nSaving all results to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print("Completed Successfully! All translations are saved.")


Saving all results to /content/drive/MyDrive/Research_Paper_Publication/test/eval/bengali_to_german_model_translation_comparison_results.csv...
Completed Successfully! All translations are saved.


In [ ]:
import pandas as pd
df = pd.read_json(OUTPUT_FILE, lines=True)
df.head(3)

,id,input_text,raw_bengali,source,original_paragraph_snippet,nllb_base_translation,nllb_full_ft_translation,nllb_lora_translation,google_translation
0,1,translate Bengali to German: গাজীপুরের কালিয়া...,গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায়...,unknown,গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায়...,"In der Öl-Schalat-Eregion von Kaliyagarh, Ghaz...",500 Mitarbeiter einer Kleidungsfabrik in der U...,"In der Öl-Schalat-Eregion von Kaliyagarh, Ghaz...",500 Arbeiter einer Bekleidungsfabrik erkrankte...
1,2,translate Bengali to German: এ ঘটনায় বিক্ষোভ ...,এ ঘটনায় বিক্ষোভ করেছেন ওই কারখানার শ্রমিকেরা।,unknown,গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায়...,Die Arbeiter der Fabrik protestierten gegen de...,Die Arbeiter der Fabrik demonstrierten.,Die Arbeiter der Fabrik protestierten gegen de...,Die Fabrikarbeiter protestierten gegen diesen ...
2,3,translate Bengali to German: সফিপুর মডার্ন হাস...,সফিপুর মডার্ন হাসপাতালের জরুরি বিভাগের চিকিত্স...,unknown,গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায়...,"Der Ärzt Al Amin, Emergency Department of Sufi...","Der Ärzt Al Amin, Emergency Department des mod...","Der Ärzt Al Amin, Emergency Department of Sufi...","Al Amin, ein Arzt in der Notaufnahme des Safip..."


**Note**: We do not have a working Bengali-to-German LoRA adapter. The German translation by LoRA Fine-tuned NLLB model is just the Base NLLB model working correctly. The output of these two are exactly the same.